In [48]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from typing import TypedDict,Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage,BaseMessage,AIMessage
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_text_splitters import RecursiveCharacterTextSplitter,TextSplitter
from langgraph.types import interrupt
from langgraph.checkpoint.memory import MemorySaver

In [49]:
load_dotenv()
model = ChatOpenAI()

In [50]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [51]:
def chat_node(state:ChatState):
    
    decision = interrupt({
        "type":"approval",
        "reason":"Model is about to answer a user question",
        "question":state["messages"][-1].content,
        "instruction":"Approve this question? yes/no"
    })
    
    if decision['approved']=='no':
        return {"message":[AIMessage(content="Not Approved")]}
    
    else:
    
        messages = state['messages']
        response = model.invoke(messages)
        return {
            'messages':[response]
        }

In [52]:
graph = StateGraph(ChatState)

graph.add_node('chat',chat_node)

graph.add_edge(START,'chat')
graph.add_edge('chat',END)

check_pointer = MemorySaver()

bot = graph.compile(checkpointer=check_pointer)

In [53]:
config ={'configurable':{
    "thread_id":'12'
}}

initial_input={
    "messages":[
        ('user',"Explain gradient descent in very simple terms")
    ]
}

In [54]:
result = bot.invoke(input=initial_input,config=config)

In [55]:
message = result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'Model is about to answer a user question',
 'question': 'Explain gradient descent in very simple terms',
 'instruction': 'Approve this question? yes/no'}

In [56]:
user_input = input(f"\nBackend message -{message}\n Approve this question? (y/n): ")

In [57]:
final_result = bot.invoke(
    Command(resume={"approved":user_input}),
    config=config
)


print(final_result)

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms', additional_kwargs={}, response_metadata={}, id='d4777169-ae7e-4637-9abc-d5a9ec71d0b8')]}
